In [4]:
import sys
sys.path.insert(0, "..")
from gpbp.layers import AdmArea
from gpbp import visualisation

from optimization import jg_opt
from functools import partial

import warnings
warnings.filterwarnings(action='ignore')

## Defining the Administrative Area

In [ ]:
NUTS

In [3]:
adm_area = AdmArea(country="Timor-Leste", level=1)

Retrieving data for Timor-Leste of granularity level 1
Administrative areas for level 1:
['Aileu' 'Ainaro' 'Ambeno' 'Baucau' 'Bobonaro' 'Covalima' 'Dili' 'Ermera'
 'Lautém' 'Liquiçá' 'Manatuto' 'Manufahi' 'Viqueque']


In [4]:
adm_area.country

Country(alpha_2='TL', alpha_3='TLS', flag='🇹🇱', name='Timor-Leste', numeric='626', official_name='Democratic Republic of Timor-Leste')

In [5]:
adm_area.country_gdf

,ID_0,COUNTRY,ID_1,NAME_1,VARNAME_1,NL_NAME_1,TYPE_1,ENGTYPE_1,CC_1,HASC_1,ISO_1,geometry
0,TLS,Timor-Leste,TLS.1_1,Aileu,,,Distrito,District,,TP.AL,,"MULTIPOLYGON (((125.58068 -8.82766, 125.57981 ..."
1,TLS,Timor-Leste,TLS.2_1,Ainaro,,,Distrito,District,,TP.AN,,"MULTIPOLYGON (((125.61008 -8.93925, 125.60912 ..."
2,TLS,Timor-Leste,TLS.3_1,Ambeno,Oecussi,,Distrito,District,,TP.AM,,"MULTIPOLYGON (((124.35570 -9.48334, 124.35561 ..."
3,TLS,Timor-Leste,TLS.4_1,Baucau,,,Distrito,District,,TP.BC,,"MULTIPOLYGON (((126.45296 -8.67774, 126.45297 ..."
4,TLS,Timor-Leste,TLS.5_1,Bobonaro,,,Distrito,District,,TP.BB,,"MULTIPOLYGON (((125.42563 -9.00174, 125.42568 ..."
5,TLS,Timor-Leste,TLS.6_1,Covalima,Cova Lima,,Distrito,District,,TP.CL,,"MULTIPOLYGON (((125.07837 -9.39620, 125.07841 ..."
6,TLS,Timor-Leste,TLS.7_1,Dili,,,Distrito,District,,TP.DL,,"MULTIPOLYGON (((125.55719 -8.62418, 125.55663 ..."
7,TLS,Timor-Leste,TLS.8_1,Ermera,,,Distrito,District,,TP.ER,,"MULTIPOLYGON (((125.37266 -8.98738, 125.37241 ..."
8,TLS,Timor-Leste,TLS.9_1,Lautém,,,Distrito,District,,TP.BT,,"MULTIPOLYGON (((126.80404 -8.75507, 126.80285 ..."
9,TLS,Timor-Leste,TLS.10_1,Liquiçá,,,Distrito,District,,TP.LQ,,"MULTIPOLYGON (((125.18736 -8.75291, 125.18698 ..."


In [6]:
adm_area = AdmArea(country="Timor-Leste", level=0)

AttributeError: 'AdmArea' object has no attribute 'adm_name'

In [7]:
adm_area.get_adm_area("Baucau")

Extracting geometry for administrative area


- `self.get_adm_area` simply extracts the geometry of a specific admin area
- However it has a special treatment of `level == 0` as mentioned before. Not sure if necessary to do this.

adm_area.get_adm_area("Timor-Leste")

## Retrieving Facility and Population data

In [ ]:
adm_area.get_facilities(method="osm", tags={"building":"hospital"})
visualisation.plot_facilities(adm_area.fac_gdf)

In [ ]:
adm_area.get_population(method="world_pop")
visualisation.plot_population_heatmap(adm_area.pop_df)

## Computing potential locations for facilities

In [ ]:
adm_area.compute_potential_fac(spacing=0.05)
visualisation.plot_facilities(adm_area.pot_fac_gdf)

## Retrieving the road network

In [11]:
adm_area.get_road_network("driving")

## Prepare optimization data

In [12]:
MAPBOX_API_TOKEN = None # fill out with your own access token for mapbox strategy
DISTANCE_TYPE = "length"
pop_count, current, potential = adm_area.prepare_optimization_data(
    DISTANCE_TYPE, [2000, 5000, 10000], "driving", "osm", population_resolution=3, mapbox_access_token=MAPBOX_API_TOKEN)

/Users/operte/repos/Public-Infrastructure-Service-Access-1/gpbp/distance.py:281: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  serve_df = pd.DataFrame(index=fac_gdf["ID"].values, data=serve_dict).applymap(
/Users/operte/repos/Public-Infrastructure-Service-Access-1/gpbp/distance.py:281: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  serve_df = pd.DataFrame(index=fac_gdf["ID"].values, data=serve_dict).applymap(


In [51]:
pop_count.shape

(124500,)

In [74]:
current['length']

,Cluster_ID,ID_2000,ID_5000,ID_10000
0,0,"[105556, 105977, 105766, 104720, 105557, 10534...","[102478, 102679, 102479, 102480, 103080, 10268...","[98673, 98873, 99873, 99472, 99073, 99273, 996..."
1,1,"[6999, 7603, 7402, 7201, 7000, 3293, 3681, 348...","[1087, 3662, 5200, 3856, 3663, 3857, 3664, 385...","[1019, 1086, 956, 1020, 1087, 895, 957, 4972, ..."


- This method is the longest.
- We have to pass the mode of transport, even though it was already defined in `get_road_network`.
- the population is projected to a lower resolution grid. the user defines the number of digits that lat and long should be projected to.
- the existing facilities and potential locations for new facilities are merged into the same gdf, I'm not sure why. Then this gdf is passed into two calls of the same function `population_served` and they are once again split into existing and potential facilities. The splitting is done by index, which is not very elegant. We could replace this with a new column that labels the type of facility (potential or existing).
- `population_served` is poorly documented and it's also quite long
  - in a first step, isopolygons are computed depending on the chosen strategy.
  - OSM is done locally with networkx and osmnx
  - at a second stage, the population and isopolygons are joined but I still didn't fully get what happens to them
  - this happens for every distance passed to `prepare_optimization_data`
- the outputs are:
  -  the population per point of the low resolution grid
  -  current and potential TODO didn't get what exactly this is

## Optimize

In [13]:
CBC_SOLVER_PATH = None # fill out the solver path where the cbc executable 
BUDGET = [5, 20, 50] # budget for the optimization in terms of how many locations can be built
cbc_optimize = partial(
                    jg_opt.OpenOptimize, solver_path=CBC_SOLVER_PATH
                )
jg_opt.Solve(pop_count, current, potential, DISTANCE_TYPE, BUDGET, optimize=cbc_optimize, type='ID')

    containing a solution
    containing a solution


(       10000      5000      2000
 5   0.281553  0.220164  0.061726
 20  0.337429  0.253523  0.071299
 50  0.374805  0.273309  0.076281,
                                                 10000  \
 5                             [401, 470, 34, 351, 80]   
 20  [401, 470, 34, 351, 80, 361, 142, 486, 344, 26...   
 50  [401, 470, 34, 351, 80, 361, 142, 486, 344, 26...   
 
                                                  5000  \
 5                             [401, 362, 279, 56, 80]   
 20  [401, 362, 279, 56, 80, 400, 449, 344, 142, 20...   
 50  [401, 362, 279, 56, 80, 400, 449, 344, 142, 20...   
 
                                                  2000  
 5                           [401, 362, 279, 142, 150]  
 20  [401, 362, 279, 142, 150, 400, 80, 449, 447, 2...  
 50  [401, 362, 279, 142, 150, 400, 80, 449, 447, 2...  )